In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import optuna
except ModuleNotFoundError as e:
    raise ModuleNotFoundError("optuna is required for this notebook. Install it first: pip install optuna") from e

from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error

from sklearn.linear_model import Lasso, ElasticNet, Ridge, BayesianRidge, HuberRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=ConvergenceWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
DURATION_CAP = 960
SEED = 2
target_col = "duration_hours"

ART_DIR = Path("./artifacts_tuning_pointfix")
ART_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("./data_finall_without_TTM.csv"),
    Path("/mnt/data/data_finall_without_TTM.csv"),
]
DATA_PATH = None
for candidate in DATA_CANDIDATES:
    if candidate.exists():
        DATA_PATH = candidate
        break
if DATA_PATH is None:
    raise FileNotFoundError("data_finall_without_TTM.csv not found in current directory or /mnt/data")

data_finall = pd.read_csv(DATA_PATH)

split = int(len(data_finall) * 0.8)
train_raw = data_finall.iloc[:split].copy()
test_raw  = data_finall.iloc[split:].copy()

# Основной train / validation / primary holdout: только типовые задачи
train_df = train_raw[train_raw[target_col] < DURATION_CAP].copy()
test_typical_df = test_raw[test_raw[target_col] < DURATION_CAP].copy()

# Secondary stress-test: полный будущий поток, включая выбросы
test_full_df = test_raw.copy()

Xtrain = train_df.drop(columns=[target_col])
ytrain = train_df[target_col]

Xtest_typical = test_typical_df.drop(columns=[target_col])
ytest_typical = test_typical_df[target_col]

Xtest_full  = test_full_df.drop(columns=[target_col])
ytest_full  = test_full_df[target_col]

# aliases for backward compatibility in a few blocks
Xtest = Xtest_typical
ytest = ytest_typical
X = Xtrain
y = ytrain

print(f"Train raw:                        {len(train_raw)} строк")
print(f"Train filtered < {DURATION_CAP}:              {len(train_df)} строк")
print(f"Primary holdout test < {DURATION_CAP}:        {len(test_typical_df)} строк")
print(f"Secondary full holdout:           {len(test_full_df)} строк")
print(f"Holdout outliers >= {DURATION_CAP}:          {(test_full_df[target_col] >= DURATION_CAP).sum()}")

print(f"\nXtrain shape: {Xtrain.shape}, Xtest_typical shape: {Xtest_typical.shape}, Xtest_full shape: {Xtest_full.shape}")
print(f"Фичи ({X.shape[1]}): {list(X.columns[:15])}{'...' if X.shape[1] > 15 else ''}")

In [ ]:
pipelines = []

pipelines.append(("Scaled_Lasso", Pipeline([("Scaler", StandardScaler()), ("Lasso", Lasso(random_state=SEED))])))
pipelines.append(("Scaled_Elastic", Pipeline([("Scaler", StandardScaler()), ("Elastic", ElasticNet(random_state=SEED))])))
pipelines.append(("Scaled_Ridge", Pipeline([("Scaler", StandardScaler()), ("Ridge", Ridge(random_state=SEED))])))
pipelines.append(("Scaled_BayRidge", Pipeline([("Scaler", StandardScaler()), ("BR", BayesianRidge())])))
pipelines.append(("Scaled_Hub-Reg", Pipeline([("Scaler", StandardScaler()), ("Hub-Reg", HuberRegressor())])))

# Для деревьев scaler не нужен
# GBoost baseline intentionally aligned with notebook 1 (default loss="squared_error")
pipelines.append(("Scaled_Gboost-Reg", GradientBoostingRegressor(random_state=SEED)))
pipelines.append((
    "Scaled_XGB_reg",
    XGBRegressor(
        objective="reg:absoluteerror",
        eval_metric="mae",
        random_state=SEED,
        tree_method="hist",
        n_jobs=-1,
    )
))

models_to_compare = [
    "BayRidge",
    "Lasso",
    "Elastic",
    "Ridge",
    "Hub-Reg",
    "Gboost-Reg",
    "XGB_reg",
]

pipe_map = {name.replace("Scaled_", ""): pipe for name, pipe in pipelines}

def reg_metrics(y_true, pred):
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "MdAE": median_absolute_error(y_true, pred),
    }

def normalize_params(params):
    clean = {}
    for k, v in params.items():
        clean[k.split("__")[-1]] = v.item() if hasattr(v, "item") else v
    return clean



def extract_core_estimator(estimator):
    return estimator.steps[-1][1] if isinstance(estimator, Pipeline) else estimator

def extract_model_params(estimator):
    core = extract_core_estimator(estimator)
    clean = {}
    for k, v in core.get_params(deep=False).items():
        clean[k] = v.item() if hasattr(v, "item") else v
    return clean

def evaluate_estimator(estimator):
    pred_typ = estimator.predict(Xtest_typical)
    pred_full = estimator.predict(Xtest_full)
    return reg_metrics(ytest_typical, pred_typ), reg_metrics(ytest_full, pred_full)

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)

## GridSearchCV — точечный первичный перебор вокруг разумных диапазонов

- Выбор лучшей конфигурации идёт по `CV MAE` на типовых задачах (`y < 960`).
- `Primary holdout` на типовых задачах считается отдельно и служит sanity-check после CV.
- `Full holdout` считается только как дополнительный stress-test и не участвует в выборе победителя.

In [ ]:
tscv = TimeSeriesSplit(n_splits=3)

param_grids = {
    "Lasso": {
        "Lasso__alpha": sorted(set(np.round(np.concatenate([np.logspace(-3, 1, 16), [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]]), 6).tolist())),
        "Lasso__selection": ["cyclic", "random"],
        "Lasso__max_iter": [100000],
    },
    "Elastic": {
        "Elastic__alpha": sorted(set(np.round(np.concatenate([np.logspace(-3, 1, 14), [0.3, 0.5, 0.7, 1.0, 1.5, 2.0]]), 6).tolist())),
        "Elastic__l1_ratio": [0.70, 0.85, 0.90, 0.95, 0.98, 0.995],
        "Elastic__max_iter": [100000],
    },
    "Ridge": {
        "Ridge__alpha": np.logspace(-2, 3, 20).tolist(),
    },
    "BayRidge": {
        "BR__alpha_1": [1e-8, 1e-6, 1e-4, 1e-2],
        "BR__alpha_2": [1e-8, 1e-6, 1e-4, 1e-2],
        "BR__lambda_1": [1e-8, 1e-6, 1e-4, 1e-2],
        "BR__lambda_2": [1e-8, 1e-6, 1e-4, 1e-2],
    },
    "Hub-Reg": {
        "Hub-Reg__epsilon": [1.1, 1.2, 1.35, 1.5, 1.75, 2.0, 2.5],
        "Hub-Reg__alpha": [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0],
        "Hub-Reg__max_iter": [10000],
    },
    "Gboost-Reg": [
        {
            "loss": ["squared_error", "absolute_error"],
            "n_estimators": [100, 150],
            "learning_rate": [0.05, 0.10],
            "max_depth": [2, 3, 4],
            "min_samples_leaf": [1, 4],
            "min_samples_split": [2, 8],
            "subsample": [0.70, 1.0],
            "max_features": [None, 1.0],
        },
        {
            "loss": ["huber"],
            "alpha": [0.90],
            "n_estimators": [100, 150],
            "learning_rate": [0.05, 0.10],
            "max_depth": [2, 3, 4],
            "min_samples_leaf": [1, 4],
            "min_samples_split": [2, 8],
            "subsample": [0.70, 1.0],
            "max_features": [None, 1.0],
        },
    ],
    "XGB_reg": {
        "n_estimators": [300, 600],
        "learning_rate": [0.02, 0.05],
        "max_depth": [2, 3],
        "min_child_weight": [5, 10],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "reg_alpha": [0.0, 0.1],
        "reg_lambda": [1.0, 5.0],
        "gamma": [0.0, 1.0],
        "objective": ["reg:absoluteerror"],
        "eval_metric": ["mae"],
        "tree_method": ["hist"],
        "random_state": [SEED],
        "n_jobs": [-1],
    },
}

grid_results = {}
grid_rows = []

for m in models_to_compare:
    t0 = time.time()
    pipe = pipe_map[m]

    baseline = clone(pipe)
    baseline.fit(Xtrain, ytrain)
    baseline_typ, baseline_full = evaluate_estimator(baseline)

    n_combos = len(list(ParameterGrid(param_grids[m])))
    print(f"\n===== {m} ({n_combos} combinations × 3 folds) =====")

    grid = GridSearchCV(
        estimator=clone(pipe),
        param_grid=param_grids[m],
        scoring="neg_mean_absolute_error",
        cv=tscv,
        n_jobs=-1,
        verbose=1,
        refit=True,
    )
    grid.fit(Xtrain, ytrain)

    tuned_typ, tuned_full = evaluate_estimator(grid.best_estimator_)
    cv_mae = -grid.best_score_
    elapsed = time.time() - t0

    grid_results[m] = {
        "baseline_estimator": baseline,
        "grid_estimator": grid.best_estimator_,
        "baseline_typical": baseline_typ,
        "baseline_full": baseline_full,
        "grid_typical": tuned_typ,
        "grid_full": tuned_full,
        "cv_mae": cv_mae,
        "best_params": extract_model_params(grid.best_estimator_),
    }

    grid_rows.append({
        "model": m,
        "baseline_typical_MAE": round(baseline_typ["MAE"], 2),
        "baseline_full_MAE": round(baseline_full["MAE"], 2),
        "grid_cv_MAE": round(cv_mae, 2),
        "grid_typical_MAE": round(tuned_typ["MAE"], 2),
        "grid_full_MAE": round(tuned_full["MAE"], 2),
        "delta_typical_vs_baseline": round(baseline_typ["MAE"] - tuned_typ["MAE"], 2),
        "best_params": extract_model_params(grid.best_estimator_),
    })

    cv_dump = pd.DataFrame(grid.cv_results_)[["params", "mean_test_score", "std_test_score", "rank_test_score"]].copy()
    cv_dump["mean_test_MAE"] = -cv_dump["mean_test_score"]
    cv_dump = cv_dump.sort_values("rank_test_score")
    cv_dump.to_csv(ART_DIR / f"grid_cv_{m}.csv", index=False)

    print(f"  Grid CV MAE (primary):   {cv_mae:.2f}")
    print(f"  Holdout MAE typical:     {tuned_typ['MAE']:.2f}")
    print(f"  Holdout MAE full:        {tuned_full['MAE']:.2f}")
    print(f"  Best params (full):      {extract_model_params(grid.best_estimator_)}")
    print(f"  Time:                    {elapsed:.1f}s")

grid_summary = pd.DataFrame(grid_rows).sort_values(["grid_cv_MAE", "grid_typical_MAE"]).reset_index(drop=True)
grid_summary.to_csv(ART_DIR / "grid_summary.csv", index=False)
grid_summary

## Optuna — более умный поиск в непрерывных диапазонах

- Целевая метрика оптимизации: `CV MAE` на типовых задачах (`train_df` / `test_typical_df`).
- Holdout считается только после завершения поиска.
- Для `GBoost` и `XGB` диапазоны расширены по более полезным параметрам.
- Для `GBoost` поиск теперь явно покрывает и baseline из ноутбука 1 (`loss="squared_error"`), и сильную зону `absolute_error`-дефолтов (`n_estimators=100`, `learning_rate=0.1`, `min_samples_leaf=1`, `min_samples_split=2`, `subsample=1.0`, `max_features=None`).


In [ ]:
N_SPLITS = 3
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

N_TRIALS = {
    "BayRidge": 120,
    "Lasso": 140,
    "Elastic": 180,
    "Ridge": 120,
    "Hub-Reg": 160,
    "Gboost-Reg": 320,
    "XGB_reg": 260,
}

print("Всего trial'ов:", sum(N_TRIALS.values()))

baseline_models = {
    "BayRidge": Pipeline([("scaler", StandardScaler()), ("model", BayesianRidge())]),
    "Lasso": Pipeline([("scaler", StandardScaler()), ("model", Lasso(max_iter=100000, selection="cyclic", random_state=SEED))]),
    "Elastic": Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(max_iter=100000, random_state=SEED))]),
    "Ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
    "Hub-Reg": Pipeline([("scaler", StandardScaler()), ("model", HuberRegressor(max_iter=10000))]),
    "Gboost-Reg": GradientBoostingRegressor(random_state=SEED),
    "XGB_reg": XGBRegressor(
        objective="reg:absoluteerror",
        eval_metric="mae",
        random_state=SEED,
        tree_method="hist",
        n_jobs=-1,
    ),
}

def build_model(model_name, trial):
    if model_name == "BayRidge":
        return Pipeline([("scaler", StandardScaler()), ("model", BayesianRidge(
            max_iter=trial.suggest_int("max_iter", 200, 1200, step=100),
            tol=trial.suggest_float("tol", 1e-5, 1e-2, log=True),
            alpha_1=trial.suggest_float("alpha_1", 1e-8, 1e-2, log=True),
            alpha_2=trial.suggest_float("alpha_2", 1e-8, 1e-2, log=True),
            lambda_1=trial.suggest_float("lambda_1", 1e-8, 1e-2, log=True),
            lambda_2=trial.suggest_float("lambda_2", 1e-8, 1e-2, log=True),
        ))])

    elif model_name == "Lasso":
        return Pipeline([("scaler", StandardScaler()), ("model", Lasso(
            alpha=trial.suggest_float("alpha", 1e-3, 10.0, log=True),
            selection=trial.suggest_categorical("selection", ["cyclic", "random"]),
            max_iter=100000,
            random_state=SEED
        ))])

    elif model_name == "Elastic":
        return Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(
            alpha=trial.suggest_float("alpha", 1e-3, 10.0, log=True),
            l1_ratio=trial.suggest_float("l1_ratio", 0.60, 0.999),
            max_iter=100000,
            random_state=SEED
        ))])

    elif model_name == "Ridge":
        return Pipeline([("scaler", StandardScaler()), ("model", Ridge(
            alpha=trial.suggest_float("alpha", 1e-2, 1e3, log=True)
        ))])

    elif model_name == "Hub-Reg":
        return Pipeline([("scaler", StandardScaler()), ("model", HuberRegressor(
            epsilon=trial.suggest_float("epsilon", 1.05, 2.5),
            alpha=trial.suggest_float("alpha", 1e-5, 5.0, log=True),
            max_iter=10000
        ))])

    elif model_name == "Gboost-Reg":
        loss = trial.suggest_categorical("loss", ["squared_error", "absolute_error", "huber"])
        params = {
            "loss": loss,
            "n_estimators": trial.suggest_int("n_estimators", 60, 320, step=10),
            "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
            "max_depth": trial.suggest_int("max_depth", 1, 4),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 24),
            "subsample": trial.suggest_float("subsample", 0.60, 1.0),
            "max_features": trial.suggest_categorical("max_features", [None, 0.70, 0.85, 1.0]),
            "random_state": SEED,
        }
        if loss == "huber":
            params["alpha"] = trial.suggest_float("alpha", 0.85, 0.95)
        return GradientBoostingRegressor(**params)

    elif model_name == "XGB_reg":
        return XGBRegressor(
            objective="reg:absoluteerror",
            eval_metric="mae",
            random_state=SEED,
            tree_method="hist",
            n_jobs=-1,
            n_estimators=trial.suggest_int("n_estimators", 200, 1200, step=50),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            max_depth=trial.suggest_int("max_depth", 2, 4),
            min_child_weight=trial.suggest_int("min_child_weight", 3, 20),
            subsample=trial.suggest_float("subsample", 0.50, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.50, 1.0),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-5, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-2, 50.0, log=True),
            gamma=trial.suggest_float("gamma", 0.0, 5.0),
            max_bin=trial.suggest_int("max_bin", 128, 512, step=64),
        )

    else:
        raise ValueError(f"Unknown model: {model_name}")

def make_objective(model_name):
    def objective(trial):
        fold_maes = []

        for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(Xtrain), start=1):
            Xtr = Xtrain.iloc[tr_idx]
            ytr = ytrain.iloc[tr_idx]
            Xva = Xtrain.iloc[va_idx]
            yva = ytrain.iloc[va_idx]

            est = build_model(model_name, trial)
            est.fit(Xtr, ytr)
            pred = est.predict(Xva)

            mae = mean_absolute_error(yva, pred)
            fold_maes.append(mae)

            trial.report(float(np.mean(fold_maes)), step=fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(fold_maes))
    return objective

studies = {}
best_estimators = {}
rows = []

for model_name in models_to_compare:
    print("=" * 100)
    print(f"Optuna | {model_name}")
    print("=" * 100)

    t0 = time.time()

    baseline = clone(baseline_models[model_name])
    baseline.fit(Xtrain, ytrain)
    baseline_typ, baseline_full = evaluate_estimator(baseline)

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=20, n_warmup_steps=1),
    )

    study.optimize(
        make_objective(model_name),
        n_trials=N_TRIALS[model_name],
        show_progress_bar=True,
        gc_after_trial=True,
        n_jobs=1,
    )

    best_model = build_model(model_name, optuna.trial.FixedTrial(study.best_params))
    best_model.fit(Xtrain, ytrain)
    tuned_typ, tuned_full = evaluate_estimator(best_model)

    elapsed = time.time() - t0

    studies[model_name] = study
    best_estimators[model_name] = best_model

    rows.append({
        "model": model_name,
        "baseline_typical_MAE": round(baseline_typ["MAE"], 2),
        "baseline_full_MAE": round(baseline_full["MAE"], 2),
        "optuna_cv_MAE": round(study.best_value, 2),
        "optuna_typical_MAE": round(tuned_typ["MAE"], 2),
        "optuna_full_MAE": round(tuned_full["MAE"], 2),
        "delta_typical_vs_baseline": round(baseline_typ["MAE"] - tuned_typ["MAE"], 2),
        "n_trials": N_TRIALS[model_name],
        "time_sec": round(elapsed, 1),
        "best_params": extract_model_params(best_model),
    })

    trials_df = study.trials_dataframe()
    trials_df.to_csv(ART_DIR / f"optuna_trials_{model_name}.csv", index=False)

    print(f"Optuna CV MAE (primary):  {study.best_value:.2f}")
    print(f"Holdout MAE typical:      {tuned_typ['MAE']:.2f}")
    print(f"Holdout MAE full:         {tuned_full['MAE']:.2f}")
    print(f"Best params (full):      {extract_model_params(best_model)}")
    print(f"Time:                    {elapsed:.1f}s")
    print()

optuna_summary = pd.DataFrame(rows).sort_values(["optuna_cv_MAE", "optuna_typical_MAE"]).reset_index(drop=True)
optuna_summary.to_csv(ART_DIR / "optuna_summary.csv", index=False)
optuna_summary

## Fine-Tuning — дожимаем несколько сильных моделей, но выбираем по CV, а не по holdout

- Для `Gboost-Reg`, `Hub-Reg`, `Elastic` stage-2 стартует от лучшей конфигурации stage-1.
- Победитель stage-2 определяется по улучшению `CV MAE` на типовых задачах.
- `Primary holdout` и `full holdout` считаются только как итоговая диагностика после выбора.

In [ ]:
tscv = TimeSeriesSplit(n_splits=3)

N_TRIALS_S2 = {
    "Gboost-Reg": 220,
    "Hub-Reg": 120,
    "Elastic": 120,
}

current_best_models = {}
current_best_params = {}
current_best_cv = {}
current_best_source = {}

for model_name in N_TRIALS_S2:
    grid_cv = grid_results[model_name]["cv_mae"]
    optuna_cv = studies[model_name].best_value

    if grid_cv <= optuna_cv:
        current_best_models[model_name] = clone(grid_results[model_name]["grid_estimator"])
        current_best_params[model_name] = extract_model_params(grid_results[model_name]["grid_estimator"])
        current_best_cv[model_name] = grid_cv
        current_best_source[model_name] = "grid"
    else:
        current_best_models[model_name] = clone(best_estimators[model_name])
        current_best_params[model_name] = extract_model_params(best_estimators[model_name])
        current_best_cv[model_name] = optuna_cv
        current_best_source[model_name] = "optuna"

print("Stage-2 starts from:")
for model_name in N_TRIALS_S2:
    print(f"  {model_name:<12s} | source={current_best_source[model_name]:<6s} | CV MAE={current_best_cv[model_name]:.2f} | params={current_best_params[model_name]}")

def build_stage2_model(model_name, trial):
    p = current_best_params[model_name]

    if model_name == "Gboost-Reg":
        loss = p.get("loss", "squared_error")
        base_n_estimators = int(p.get("n_estimators", 100))
        base_learning_rate = float(p.get("learning_rate", 0.10))
        base_depth = int(p.get("max_depth", 3))
        base_leaf = int(p.get("min_samples_leaf", 1))
        base_split = int(p.get("min_samples_split", 2))
        base_subsample = float(p.get("subsample", 1.0))
        base_max_features = p.get("max_features", None)

        params = {
            "loss": loss,
            "n_estimators": trial.suggest_int("n_estimators", max(60, base_n_estimators - 50), min(360, base_n_estimators + 120), step=10),
            "learning_rate": trial.suggest_float("learning_rate", max(0.03, base_learning_rate * 0.6), min(0.18, base_learning_rate * 1.5)),
            "max_depth": trial.suggest_int("max_depth", max(1, base_depth - 1), min(5, base_depth + 1)),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", max(1, base_leaf - 2), base_leaf + 8),
            "min_samples_split": trial.suggest_int("min_samples_split", max(2, base_split - 2), base_split + 10),
            "subsample": trial.suggest_float("subsample", max(0.60, base_subsample - 0.15), min(1.0, base_subsample + 0.10)),
            "random_state": SEED,
        }

        if base_max_features is None or (isinstance(base_max_features, float) and np.isnan(base_max_features)):
            params["max_features"] = trial.suggest_categorical("max_features", [None, 0.85, 1.0])
        else:
            params["max_features"] = trial.suggest_float(
                "max_features",
                max(0.60, float(base_max_features) - 0.15),
                min(1.0, float(base_max_features) + 0.10),
            )

        if loss == "huber":
            params["alpha"] = trial.suggest_float("alpha", max(0.85, float(p.get("alpha", 0.90)) - 0.05), min(0.98, float(p.get("alpha", 0.90)) + 0.05))

        return GradientBoostingRegressor(**params)

    elif model_name == "Hub-Reg":
        return Pipeline([("scaler", StandardScaler()), ("model", HuberRegressor(
            epsilon=trial.suggest_float("epsilon", max(1.05, float(p.get("epsilon", 1.35)) - 0.15), min(2.5, float(p.get("epsilon", 1.35)) + 0.25)),
            alpha=trial.suggest_float("alpha", max(1e-5, float(p.get("alpha", 0.01)) / 3), min(10.0, max(1e-4, float(p.get("alpha", 0.01)) * 3)), log=True),
            max_iter=10000
        ))])

    elif model_name == "Elastic":
        return Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(
            alpha=trial.suggest_float("alpha", max(1e-4, float(p.get("alpha", 0.5)) / 3), min(10.0, float(p.get("alpha", 0.5)) * 3), log=True),
            l1_ratio=trial.suggest_float("l1_ratio", max(0.50, float(p.get("l1_ratio", 0.95)) - 0.10), min(0.9999, float(p.get("l1_ratio", 0.95)) + 0.05)),
            selection=trial.suggest_categorical("selection", ["cyclic", "random"]),
            max_iter=100000,
            random_state=SEED
        ))])

    else:
        raise ValueError(model_name)

def make_objective_s2(model_name):
    def objective(trial):
        fold_maes = []

        for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(Xtrain), start=1):
            Xtr = Xtrain.iloc[tr_idx]
            ytr = ytrain.iloc[tr_idx]
            Xva = Xtrain.iloc[va_idx]
            yva = ytrain.iloc[va_idx]

            est = build_stage2_model(model_name, trial)
            est.fit(Xtr, ytr)
            pred = est.predict(Xva)

            mae = mean_absolute_error(yva, pred)
            fold_maes.append(mae)

            trial.report(float(np.mean(fold_maes)), step=fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(fold_maes))
    return objective

stage2_rows = []
stage2_studies = {}
stage2_best_estimators = {}
stage2_final_params = {}
stage2_final_cv = {}
stage2_final_source = {}

for model_name in N_TRIALS_S2:
    print("=" * 95)
    print(f"Stage-2 Optuna | {model_name}")
    print("=" * 95)

    t0 = time.time()

    baseline = clone(current_best_models[model_name])
    baseline.fit(Xtrain, ytrain)
    baseline_typ, baseline_full = evaluate_estimator(baseline)
    baseline_cv = current_best_cv[model_name]

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED, warn_independent_sampling=False),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=20, n_warmup_steps=1),
    )

    study.optimize(
        make_objective_s2(model_name),
        n_trials=N_TRIALS_S2[model_name],
        show_progress_bar=True,
        gc_after_trial=True,
        n_jobs=1,
    )

    tuned_model = build_stage2_model(model_name, optuna.trial.FixedTrial(study.best_params))
    tuned_model.fit(Xtrain, ytrain)
    tuned_typ, tuned_full = evaluate_estimator(tuned_model)

    if study.best_value < baseline_cv:
        winner = "stage2_optuna"
        final_estimator = tuned_model
        final_params = extract_model_params(tuned_model)
        final_cv = study.best_value
        final_typ, final_full = tuned_typ, tuned_full
    else:
        winner = "stage1_keep"
        final_estimator = baseline
        final_params = extract_model_params(baseline)
        final_cv = baseline_cv
        final_typ, final_full = baseline_typ, baseline_full

    elapsed = time.time() - t0

    stage2_studies[model_name] = study
    stage2_best_estimators[model_name] = final_estimator
    stage2_final_params[model_name] = final_params
    stage2_final_cv[model_name] = final_cv
    stage2_final_source[model_name] = winner

    trials_df = study.trials_dataframe()
    trials_df.to_csv(ART_DIR / f"stage2_trials_{model_name}.csv", index=False)

    stage2_rows.append({
        "model": model_name,
        "stage1_source": current_best_source[model_name],
        "stage1_cv_MAE": round(baseline_cv, 2),
        "stage1_typical_MAE": round(baseline_typ["MAE"], 2),
        "stage1_full_MAE": round(baseline_full["MAE"], 2),
        "stage2_cv_MAE": round(study.best_value, 2),
        "stage2_typical_MAE": round(tuned_typ["MAE"], 2),
        "stage2_full_MAE": round(tuned_full["MAE"], 2),
        "winner": winner,
        "final_cv_MAE": round(final_cv, 2),
        "final_typical_MAE": round(final_typ["MAE"], 2),
        "final_full_MAE": round(final_full["MAE"], 2),
        "n_trials": N_TRIALS_S2[model_name],
        "time_sec": round(elapsed, 1),
        "stage2_best_params": extract_model_params(tuned_model),
        "final_params_to_keep": final_params,
    })

    print(f"Stage-1 CV MAE:            {baseline_cv:.2f}")
    print(f"Stage-2 CV MAE:            {study.best_value:.2f}")
    print(f"Final typical holdout MAE: {final_typ['MAE']:.2f}")
    print(f"Final full holdout MAE:    {final_full['MAE']:.2f}")
    print(f"Winner to keep:            {winner}")
    print(f"Final params to keep:      {final_params}")
    print(f"Time:                      {elapsed:.1f}s")
    print()

stage2_summary = pd.DataFrame(stage2_rows).sort_values(["final_cv_MAE", "final_typical_MAE"]).reset_index(drop=True)
stage2_summary.to_csv(ART_DIR / "stage2_summary.csv", index=False)
stage2_summary

## Интерпретация финального ranking

- Формальный победитель определяется по минимальному `CV MAE` на типовых задачах (`y < 960`).
- Если разница между 1-м и 2-м местом меньше `0.5 MAE`, это трактуется как **практически ничейный результат**: в тексте диплома корректнее говорить о двух почти равных лидерах, а не о безусловной победе одной модели.
- `Full holdout` используется только как `stress-test` и не участвует в выборе победителя.


In [ ]:
# Финальная сводка:
# - если для модели был Stage-2, берём победителя stage-2 vs stage-1 по CV;
# - иначе берём лучший из GridSearch / Optuna по CV.
# Итоговый ranking строится по CV MAE на типовых задачах.

final_rows = []

for model_name in models_to_compare:
    if model_name in stage2_best_estimators:
        final_estimator = clone(stage2_best_estimators[model_name])
        final_estimator.fit(Xtrain, ytrain)
        final_typ, final_full = evaluate_estimator(final_estimator)

        final_rows.append({
            "model": model_name,
            "selected_from": stage2_final_source.get(model_name, "stage2"),
            "cv_MAE": round(stage2_final_cv[model_name], 2),
            "holdout_typical_MAE": round(final_typ["MAE"], 2),
            "holdout_typical_RMSE": round(final_typ["RMSE"], 2),
            "holdout_typical_MdAE": round(final_typ["MdAE"], 2),
            "holdout_full_MAE": round(final_full["MAE"], 2),
            "holdout_full_RMSE": round(final_full["RMSE"], 2),
            "holdout_full_MdAE": round(final_full["MdAE"], 2),
            "params": stage2_final_params[model_name],
        })
        continue

    grid_cv = grid_results[model_name]["cv_mae"]
    optuna_cv = studies[model_name].best_value

    if grid_cv <= optuna_cv:
        final_estimator = clone(grid_results[model_name]["grid_estimator"])
        selected_from = "grid"
        cv_mae = grid_cv
        params = extract_model_params(grid_results[model_name]["grid_estimator"])
    else:
        final_estimator = clone(best_estimators[model_name])
        selected_from = "optuna"
        cv_mae = optuna_cv
        params = extract_model_params(best_estimators[model_name])

    final_estimator.fit(Xtrain, ytrain)
    final_typ, final_full = evaluate_estimator(final_estimator)

    final_rows.append({
        "model": model_name,
        "selected_from": selected_from,
        "cv_MAE": round(cv_mae, 2),
        "holdout_typical_MAE": round(final_typ["MAE"], 2),
        "holdout_typical_RMSE": round(final_typ["RMSE"], 2),
        "holdout_typical_MdAE": round(final_typ["MdAE"], 2),
        "holdout_full_MAE": round(final_full["MAE"], 2),
        "holdout_full_RMSE": round(final_full["RMSE"], 2),
        "holdout_full_MdAE": round(final_full["MdAE"], 2),
        "params": params,
    })

final_summary = pd.DataFrame(final_rows).sort_values(["cv_MAE", "holdout_typical_MAE"]).reset_index(drop=True)
final_summary.to_csv(ART_DIR / "final_summary.csv", index=False)
save_json(final_rows, ART_DIR / "final_summary.json")

print("Artifacts saved to:", ART_DIR.resolve())
final_summary

best_cv = float(final_summary["cv_MAE"].min())
near_ties = final_summary[final_summary["cv_MAE"] <= best_cv + 0.5].copy()
near_ties["cv_gap_to_best"] = (near_ties["cv_MAE"] - best_cv).round(2)
near_ties.to_csv(ART_DIR / "near_tie_candidates.csv", index=False)

if len(near_ties) > 1:
    print("\nInterpretation note:")
    print("Primary winner is chosen by minimal CV MAE on typical tasks.")
    print("At the same time, several models are within 0.5 MAE of the best CV result, so they should be treated as near-tied contenders.")
    display_cols = ["model", "selected_from", "cv_MAE", "cv_gap_to_best", "holdout_typical_MAE", "holdout_full_MAE"]
    display(near_ties[display_cols].reset_index(drop=True))
else:
    print("\nInterpretation note: the winner is separated from the runner-up by more than 0.5 MAE in CV.")
